# ⏱️ The 6-Day Battle Plan

## Day 1 & 2: Master Platform Engineer Data Structures

Forget 90% of LeetCode. Focus only on two patterns in Python: **Dictionaries (Hash Maps)** and **Linear Array/String Parsing**.

### What to practice
- **Grouping data** — parse a flat list of server logs and group errors by service name using `collections.defaultdict`
- **Counting frequencies** — use `collections.Counter`
- **O(1) lookups** — find elements instantly using Python `set`

### Mindset shift
If a question cannot be solved with a plain `dict` or a simple loop, skip it. The interviewer wants to see you manipulate configuration data and logs cleanly — not solve graph problems.

## Day 3 & 4: Python Concurrency for Databases

YugabyteDB is built for high throughput and network I/O. Platform automation is usually **I/O-bound** — waiting for database responses or API calls.

### What to memorize and write down

- **`asyncio` pattern** — write an `async` function that fetches data from 5 database nodes concurrently using `asyncio.gather()`
- **The GIL (Global Interpreter Lock)** — explain why you use `asyncio` or `threading` for database queries (I/O-bound), but `multiprocessing` for heavy computation like compressing backup tarballs (CPU-bound)
- **Connection pools** — understand how thread-safe queues or pools prevent a script from opening 10,000 simultaneous connections to YugabyteDB and crashing it

## Day 5: The "Production-Ready" Blueprint

Build confidence by applying a rigid 4-step framework to every problem. Write it out before touching the logic.

### 4-Step Framework

1. **Input Validation** — check inputs first: `if not logs: return []`
2. **Graceful Error Handling** — wrap logic in `try...except Exception as e:` and log the error cleanly
3. **Type Hinting** — use standard hints: `def parse_nodes(nodes: list[str]) -> dict[str, int]:` — this signals production-grade code immediately
4. **Resource Cleanup** — if the question mentions files or database connections, always use context managers: `with open(...) as f:`

## Day 6: Mock Interview & Big-O Speedrun

### Complexity Cheat Sheet

| Operation | Complexity |
|-----------|------------|
| Dict key lookup | O(1) |
| Set membership check | O(1) |
| Loop through a list | O(N) |
| Sort a list (`.sort()`) | O(N log N) |

### Talking Strategy

Practice explaining your code out loud as you write it. If you get stuck on syntax, say:

> "I'm going to assume this built-in returns a list, and I'll handle the logic based on that."

Interviewers value this pragmatic approach — it shows you can think and communicate under pressure.

In [1]:
"""
Wise YugabyteDB Platform Engineer Interview Preparation Suite
-------------------------------------------------------------
This script contains production-ready, idiomatic Python patterns targeting:
1. Log Parsing & State Tracking (DSA)
2. Asynchronous I/O & Connection Aggregation (Concurrency)
3. Defensive Coding, Type Hinting, and Robust Error Handling (Production-Ready)
"""

import asyncio
import logging
from collections import defaultdict
from typing import Dict, List, Set

# -------------------------------------------------------------------------
# GLOBAL LOGGING SETUP (Crucial for Production Code over print statements)
# -------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(filename)s:%(lineno)d - %(message)s"
)
logger = logging.getLogger("WisePrepSuite")


# =========================================================================
# DAYS 1 & 2: DSA - TEXT PARSING & INFRASTRUCTURE STATE TRACKING
# =========================================================================

def parse_yugabyte_logs(raw_logs: List[str]) -> Dict[str, List[str]]:
    """
    Parses flat database log streams to extract and group critical errors.
    Uses an O(N) single-pass approach via collections.defaultdict.
    
    Time Complexity: O(N) where N is the number of log lines.
    Space Complexity: O(M) where M is the unique nodes generating errors.
    """
    # 1. Input Validation / Edge Case Defense
    if not raw_logs:
        logger.warning("Empty log sequence passed to processor.")
        return {}

    # 2. Idiomatic Initialization (Prevents manual KeyError checks)
    node_error_map = defaultdict(list)

    try:
        for line in raw_logs:
            # Strip whitespace and skip empty lines or comments
            clean_line = line.strip()
            if not clean_line or clean_line.startswith("#"):
                continue
                
            # Expected pattern: "NODE_IP | STATUS | MESSAGE"
            parts = [p.strip() for p in clean_line.split("|")]
            
            # Defensive unpacking protection against corrupted log formats
            if len(parts) < 3:
                logger.debug(f"Skipping malformed log line: {clean_line}")
                continue
                
            node_ip, status, message = parts[0], parts[1], parts[2]
            
            # Filtering criteria: track only operational degradation states
            if status in {"ERROR", "CRITICAL", "FATAL"}:
                node_error_map[node_ip].append(message)
                
    except Exception as e:
        logger.error(f"Fatal exception during log parsing workflow: {str(e)}")
        raise

    return dict(node_error_map)


# =========================================================================
# DAYS 3 & 4: CONCURRENCY - ASYNC I/O HEALTH CHECKER FOR DISTRIBUTED NODES
# =========================================================================

async def mock_fetch_node_metrics(node_ip: str) -> Dict[str, str]:
    """
    Simulates an asynchronous RPC or HTTP health probe to a YugabyteDB tablet server.
    Emulates network latency without blocking the main execution thread.
    """
    logger.info(f"Initiating health check handshake for node: {node_ip}")
    
    # Non-blocking sleep simulates waiting for remote cluster network I/O
    await asyncio.sleep(0.5) 
    
    # Simple static simulator logic
    if node_ip == "10.0.0.3":
        return {"node": node_ip, "status": "UNHEALTHY", "reason": "High WAL disk lag"}
    return {"node": node_ip, "status": "HEALTHY", "reason": "Synced"}


async def audit_cluster_health_concurrently(cluster_ips: List[str]) -> List[Dict[str, str]]:
    """
    Executes multiple infrastructure health probes concurrently using asyncio.gather.
    Essential pattern for managing highly scaleable, multi-node platform operations.
    """
    if not cluster_ips:
        return []

    # Dedup inputs efficiently via Sets to prevent redundant network traffic
    unique_ips: Set[str] = set(cluster_ips)
    
    # Provision a collection of asynchronous task coroutines
    tasks = [mock_fetch_node_metrics(ip) for ip in unique_ips]
    
    try:
        # Gather all futures concurrently. return_exceptions=True prevents 
        # a single failing node from crashing the entire audit pipeline.
        results = await asyncio.gather(*tasks, return_exceptions=True)
        
        sanitised_results = []
        for res in results:
            if isinstance(res, Exception):
                logger.error(f"A concurrent probe encountered a critical failure: {res}")
                continue
            sanitised_results.append(res)
            
        return sanitised_results

    except Exception as general_err:
        logger.critical(f"Async orchestration engine mapping failed: {general_err}")
        return []


# =========================================================================
# DAYS 5 & 6: MAIN EXECUTION ENTRYPOINT (PRODUCTION CODE SANITY VERIFICATION)
# =========================================================================

def main():
    """
    Acts as the harness executing mock workflows to evaluate implementation accuracy.
    """
    logger.info("--- STARTING WORKFLOW PIPELINE VERIFICATION ---")

    # --- Test Case 1: Log Parsing Engine ---
    mock_log_stream = [
        "10.0.0.1 | INFO  | Cluster bootstrap complete",
        "10.0.0.2 | ERROR | Connection timed out to tablet peer",
        "10.0.0.1 | FATAL | Out of disk space for xcluster replication",
        "10.0.0.2 | ERROR | Tablet leader election timeout",
        "         | SKIP  | Empty or corrupt node data line",
        "# This is an infrastructure metric comment line to be skipped"
    ]
    
    logger.info("Executing Days 1-2 Log Analysis...")
    parsed_errors = parse_yugabyte_logs(mock_log_stream)
    logger.info(f"Parser Output Summary: {parsed_errors}")

    # --- Test Case 2: Asynchronous Platform Concurrency ---
    mock_infrastructure_topology = ["10.0.0.1", "10.0.0.2", "10.0.0.3", "10.0.0.1"]
    
    logger.info("Executing Days 3-4 Async Network Probe Simulation...")
    # Bootstrap the asynchronous event loop safely
    cluster_report = asyncio.run(audit_cluster_health_concurrently(mock_infrastructure_topology))
    logger.info(f"Async Audit Output Summary: {cluster_report}")

    logger.info("--- WORKFLOW PIPELINE VERIFICATION SUCCESSFUL ---")


if __name__ == "__main__":
    main()


2026-06-15 21:12:20,206 [INFO] 3986595479.py:134 - --- STARTING WORKFLOW PIPELINE VERIFICATION ---
2026-06-15 21:12:20,207 [INFO] 3986595479.py:146 - Executing Days 1-2 Log Analysis...
2026-06-15 21:12:20,207 [INFO] 3986595479.py:148 - Parser Output Summary: {'10.0.0.2': ['Connection timed out to tablet peer', 'Tablet leader election timeout'], '10.0.0.1': ['Out of disk space for xcluster replication']}
2026-06-15 21:12:20,209 [INFO] 3986595479.py:153 - Executing Days 3-4 Async Network Probe Simulation...


RuntimeError: asyncio.run() cannot be called from a running event loop